# 🧬 Panacée — Entraînement Phase 2 (Kaggle)

**Phase 2** : Fine-tuning toxicité sur Tox21 (12 endpoints, ROC-AUC supervisé cliniquement).

### Avant de lancer :
1. Démarrer votre dashboard local : `python -m webapp.run`
2. Lancer ngrok dans un 2ème terminal : `ngrok http 8000`
3. Copier l'URL ngrok (ex. `https://abc123.ngrok-free.app`) → `NGROK_URL` ci-dessous
4. Choisir le même `TOKEN` que `PANACEE_INGEST_TOKEN` dans votre `.env` local
5. Appuyer sur ▶▶ (Run All)

> Les métriques arrivent en temps réel dans le dashboard (onglet Runs + SSE).

In [ ]:
# ╔══════════════════════════════════════════════╗
# ║  CONFIGURATION — modifier ces 4 lignes ONLY  ║
# ╚══════════════════════════════════════════════╝

NGROK_URL   = "https://XXXX.ngrok-free.app"   # URL publique de votre dashboard
TOKEN       = "mon_token_secret_42"            # == PANACEE_INGEST_TOKEN dans .env local
RUN_NAME    = "kaggle_phase2_run01"            # Nom affiché dans le dashboard
N_EPOCHS    = 50                               # Nombre d'epochs (GPU: 50-100, CPU: 10-20)

# Optionnel : limiter les molécules pour un run de test rapide (None = tout Tox21)
MAX_MOLECULES = None   # ex: 2000 pour tester en 10 min

print(f"Configuration :")
print(f"  URL dashboard : {NGROK_URL}")
print(f"  Nom du run    : {RUN_NAME}")
print(f"  Epochs        : {N_EPOCHS}")
print(f"  Max molécules : {MAX_MOLECULES or 'toutes'}")

In [ ]:
# ── Vérification de la connexion avec le dashboard ──────────────────────────
import urllib.request, json

health_url = NGROK_URL.rstrip('/') + '/api/health'
try:
    r = urllib.request.urlopen(health_url, timeout=8)
    data = json.loads(r.read())
    print(f"✅ Dashboard joignable — {data}")
except Exception as e:
    print(f"❌ Dashboard NON joignable ({health_url})")
    print(f"   Erreur : {e}")
    print()
    print("Checklist :")
    print("  1. Dashboard lancé ? → python -m webapp.run")
    print("  2. Ngrok lancé ?     → ngrok http 8000")
    print("  3. NGROK_URL correct ? (copier depuis le terminal ngrok)")
    raise SystemExit("Corrigez la connexion avant de continuer.")

In [ ]:
# ── Variables d'environnement (AVANT tout import du projet) ─────────────────
import os

os.environ["PANACEE_PUSH_URL"]   = NGROK_URL
os.environ["PANACEE_PUSH_TOKEN"] = TOKEN
os.environ["PANACEE_PUSH_RUN"]   = RUN_NAME
os.environ["PYTHONIOENCODING"]   = "utf-8"

print("Variables d'environnement configurées ✓")

In [ ]:
# ── Installation des dépendances ────────────────────────────────────────────
# Kaggle a déjà PyTorch. On installe le reste.
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

print("Installation de torch-geometric...")
pip("torch-geometric")

print("Installation de rdkit...")
pip("rdkit")

print("Installation de deepchem (pour télécharger Tox21)...")
pip("deepchem")

print("Dépendances OK ✓")

In [ ]:
# ── Cloner le repo Panacée ──────────────────────────────────────────────────
import sys
from pathlib import Path

REPO_URL  = "https://github.com/jumoras0000/savh_gnn.git"
CLONE_DIR = Path("/kaggle/working/panacee")

if not CLONE_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)], check=True)
    print("Repo cloné ✓")
else:
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull", "--ff-only"], check=False)
    print("Repo mis à jour ✓")

PROJET = CLONE_DIR / "Projet_Panacee"
if str(PROJET) not in sys.path:
    sys.path.insert(0, str(PROJET))

print(f"Projet prêt : {PROJET}")

In [ ]:
# ── Répertoire de sauvegarde (dans /kaggle/working pour persistance) ────────
SAVE_DIR = f"/kaggle/working/checkpoints/{RUN_NAME}"
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
print(f"Checkpoints → {SAVE_DIR}")

In [ ]:
# ── Lancer l'entraînement Phase 2 ──────────────────────────────────────────
# sys.argv est utilisé par run_phase2.main() pour parser les args.
import os as _os
_os.chdir(str(PROJET))   # run_phase2 résout les chemins depuis PROJECT_ROOT

sys.argv = [
    "run_phase2.py",
    "--download",                        # Télécharger Tox21 automatiquement
    "--save_dir",  SAVE_DIR,
    "--epochs",    str(N_EPOCHS),
]
if MAX_MOLECULES:
    sys.argv += ["--max_molecules", str(MAX_MOLECULES)]

print("Démarrage Phase 2...")
print(f"  Commande : {' '.join(sys.argv)}")
print(f"  Push URL  : {os.environ.get('PANACEE_PUSH_URL')}")
print()

from run_phase2 import main
main()

In [ ]:
# ── Résumé final + instructions de téléchargement ──────────────────────────
import glob

pth_files = glob.glob(f"{SAVE_DIR}/**/*.pth", recursive=True)
print("Checkpoints sauvegardés :")
for f in sorted(pth_files):
    size_mb = Path(f).stat().st_size / 1e6
    print(f"  {f}  ({size_mb:.1f} MB)")

print()
print("Pour télécharger le meilleur checkpoint :")
print("  1. Onglet Output (à droite) → télécharger best_toxicity_model.pth")
print("  2. Ou exporter via dataset Kaggle pour la Phase 3")
print()
print("Pour lancer la Phase 3 depuis ce checkpoint :")
print("  python run_phase3.py --pretrained_model <path>/best_toxicity_model.pth --download")